# Update log
2024/08/29
- Test without Responsivity and/or baseline
- Test with reduced number of features
- Baseline alone gives up to 75% accuracy
- Suspect data distribution due to always running experiment in 0,1,2,3,4
- Run experiment in De Bruijn sequence to balance the adjacent experiment channels
---

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import json

spk_data = "D:\\code\\uom_explore\\model_input\\3_features.csv"
spk_pca_data = "D:\\code\\uom_explore\\model_input\\pca_df.csv"
spk_20 = "D:\\code\\uom_explore\\data_science\\reduced\\20_140_195_df.csv"
spk_full = "D:\\code\\uom_explore\\data_science\\df.csv"
debruijn_1 = "D:\\code\\uom_explore\\processed_data\\metrics_exp_brujin_seq_1.csv"

hkr_wsl_data = "/home/hk-wsl/code/uom_explore/model_input/feature_matrix.csv"
hkr_pca_data = "/home/hk-wsl/code/uom_explore/model_input/feature_pca.csv"

spk_json = "/home/gavinlouuu/coding/uom_explore/data_science/parameter.json"
hkr_wsl_json = "/home/hk-wsl/code/uom_explore/data_science/parameter.json"

data_path = debruijn_1
param_path = spk_json

with open('parameter.json','r') as file:
    params = json.load(file)

# Hyperparameters
# Extract parameters from the JSON object
hidden_size = params['mlp']['hidden_size']
ground_truth = params['ground_truth']
num_epochs = params['mlp']['num_epochs']
batch_size = params['mlp']['batch_size']
learning_rate = params['mlp']['learning_rate']
momentum_value = params['mlp']['momentum_value']
dropout_rate = params['mlp']['dropout']
weight_decay = params['mlp']['weight_decay']
random_state = params['random_state']

scheduler_params = params['mlp']['scheduler']

df = pd.read_csv(data_path)
# drop experiment_id column
# df.drop('experiment_id', axis=1, inplace=True)
print(type(df))
df.head()



<class 'pandas.core.frame.DataFrame'>


,experiment_id,channel_id,baseline_160,baseline_162,baseline_165,baseline_167,baseline_170,baseline_172,baseline_175,baseline_177,...,temperature_max,temperature_std,humidity_mean,humidity_min,humidity_max,humidity_std,pressure_mean,pressure_min,pressure_max,pressure_std
0,20240829145200s1c0r0,0,15333.197116,22957.351691,23848.697631,23071.544934,21668.667869,20477.232045,19239.758925,18185.434345,...,33.21,0.333116,64.992000,60.89,68.48,2.459250,100210.552941,100201.0,100213.0,1.802814
1,20240829145235s1c1r0,1,21815.038599,33226.561073,34565.623032,33401.419655,31172.487756,29492.245969,27552.348528,25827.395031,...,33.29,0.036399,62.682083,60.15,65.88,1.926198,100210.319444,100209.0,100212.0,0.667693
2,20240829145311s1c2r0,2,28245.293614,44118.401113,45736.157153,43995.030951,41154.533173,38596.182230,35870.663615,33576.358547,...,33.40,0.039994,59.122069,58.56,59.73,0.344570,100210.344828,100208.0,100212.0,0.899969
3,20240829145346s1c3r0,3,32950.229651,52038.602759,54191.269085,51752.884670,47946.761953,45108.761385,42011.398789,39147.586067,...,33.44,0.029688,59.461351,58.47,60.65,0.728982,100207.351351,100204.0,100209.0,1.127884
4,20240829145422s1c4r0,4,18762.637612,28167.472573,28372.246360,26803.690386,24567.917448,22757.383722,20645.451620,19187.641379,...,33.49,0.029308,60.926180,58.38,64.68,2.028016,100207.573034,100205.0,100210.0,1.185975


## Load all features

In [2]:
# Get all column names from the DataFrame
all_columns = df.columns.tolist()

# Remove 'channel_id' and the ground truth from the list of features
features = [col for col in all_columns if col != 'experiment_id' and col != ground_truth]

# Print the features
print("Features:")
print(json.dumps(features, indent=2))



Features:
[
  "baseline_160",
  "baseline_162",
  "baseline_165",
  "baseline_167",
  "baseline_170",
  "baseline_172",
  "baseline_175",
  "baseline_177",
  "baseline_180",
  "baseline_182",
  "baseline_185",
  "baseline_187",
  "baseline_190",
  "baseline_192",
  "baseline_195",
  "baseline_197",
  "baseline_200",
  "baseline_202",
  "baseline_205",
  "baseline_210",
  "baseline_212",
  "baseline_215",
  "baseline_217",
  "baseline_220",
  "baseline_222",
  "baseline_225",
  "baseline_227",
  "baseline_230",
  "baseline_232",
  "baseline_235",
  "baseline_237",
  "max_reaction_R_160",
  "max_reaction_R_162",
  "max_reaction_R_165",
  "max_reaction_R_167",
  "max_reaction_R_170",
  "max_reaction_R_172",
  "max_reaction_R_175",
  "max_reaction_R_177",
  "max_reaction_R_180",
  "max_reaction_R_182",
  "max_reaction_R_185",
  "max_reaction_R_187",
  "max_reaction_R_190",
  "max_reaction_R_192",
  "max_reaction_R_195",
  "max_reaction_R_197",
  "max_reaction_R_200",
  "max_reaction_R_202"

# Keep all features

In [3]:
# X includes all features
X = df[features]



## Select settings to keep

In [4]:
# # Extract all unique numbers from feature names
# feature_numbers = set()
# for feature in features:
#     parts = feature.split('_')
#     if len(parts) > 1 and parts[-1].isdigit():
#         feature_numbers.add(int(parts[-1]))

# # Sort the numbers
# sorted_numbers = sorted(feature_numbers)

# # Select settings to keep
# # settings_to_keep = [140, 150, 152, 155, 157, 160, 162, 165, 167, 170, 172, 175, 177, 180, 182, 185, 187, 190, 192, 195]
# settings_to_keep = [140]

# # Filter the features to keep only the selected settings
# features_to_keep = [feature for feature in features if int(feature.split('_')[-1]) in settings_to_keep]

# # Update the features list
# features = features_to_keep

# # Print the features
# print("Features:")
# print(json.dumps(features, indent=2))


## Select features to keep

In [5]:
# Remove features with _min, _max, and _std suffixes
features_to_keep = [col for col in features if not any(suffix in col for suffix in ['_std'])]

# Update the features list
features = features_to_keep

# Update X dataframe to only include the kept features
X = df[features]

# Update input_size
input_size = len(features)

# print(f"Features after removing '_min', '_max', and '_std' suffixes:")
# print(json.dumps(features, indent=2))
# print(f"New input size: {input_size}")



# Data split and scale

In [6]:
# Preview features
X = X[features]
print(X.head())


   baseline_160  baseline_162  baseline_165  baseline_167  baseline_170  \
0  15333.197116  22957.351691  23848.697631  23071.544934  21668.667869   
1  21815.038599  33226.561073  34565.623032  33401.419655  31172.487756   
2  28245.293614  44118.401113  45736.157153  43995.030951  41154.533173   
3  32950.229651  52038.602759  54191.269085  51752.884670  47946.761953   
4  18762.637612  28167.472573  28372.246360  26803.690386  24567.917448   

   baseline_172  baseline_175  baseline_177  baseline_180  baseline_182  ...  \
0  20477.232045  19239.758925  18185.434345  17036.049028  16088.438761  ...   
1  29492.245969  27552.348528  25827.395031  23945.838276  22648.643782  ...   
2  38596.182230  35870.663615  33576.358547  31041.096550  29270.917883  ...   
3  45108.761385  42011.398789  39147.586067  36154.896841  33832.908399  ...   
4  22757.383722  20645.451620  19187.641379  17621.829733  16292.075230  ...   

   responsivity_237  temperature_mean  temperature_min  temperature_

In [7]:
input_size = len(X.columns)  # removing the ground truth from the number of columns counted
num_classes = df[ground_truth].nunique()
print(f"Number of classes: {num_classes}")
print(f"Number of features: {input_size}")

# Preview ground truth
y = df[ground_truth]

# Split into training, validation, and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=random_state)  # This makes 60%, 20%, 20%

# Initialize the StandardScaler
scaler = StandardScaler()
# scaler = MinMaxScaler(feature_range=(0,255)) # 

# Fit the scaler to the training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to validation and test sets
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert arrays to tensors
X_train_scaled = torch.tensor(X_train_scaled, dtype=torch.float32)#.unsqueeze(1)  # Shape: [batch_size, 1, num_features]
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.long)  # Convert to NumPy array first
X_val_scaled = torch.tensor(X_val_scaled, dtype=torch.float32)#.unsqueeze(1)  # Shape: [batch_size, 1, num_features]
y_val = torch.tensor(y_val.to_numpy(), dtype=torch.long)  # Convert to NumPy array first
X_test_scaled = torch.tensor(X_test_scaled, dtype=torch.float32)#.unsqueeze(1)  # Shape: [batch_size, 1, num_features]
y_test = torch.tensor(y_test.to_numpy(), dtype=torch.long)  # Convert to NumPy array first

# Create datasets
train_dataset = TensorDataset(X_train_scaled, y_train)
val_dataset = TensorDataset(X_val_scaled, y_val)
test_dataset = TensorDataset(X_test_scaled, y_test)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Number of classes: 5
Number of features: 102


In [8]:
def augment_data(X, y, noise_factor=0.05, num_augmentations=1):
    augmented_X = [X]
    augmented_y = [y]
    
    for _ in range(num_augmentations):
        noise = torch.randn_like(X) * noise_factor
        X_augmented = X + noise
        augmented_X.append(X_augmented)
        augmented_y.append(y)
    
    return torch.cat(augmented_X), torch.cat(augmented_y)

# Use the augmentation in your data preparation
if params['mlp']['augmentation']['enabled']:
    X_train_augmented, y_train_augmented = augment_data(
        X_train_scaled, 
        y_train, 
        noise_factor=params['mlp']['augmentation']['noise_factor'],
        num_augmentations=params['mlp']['augmentation']['num_augmentations']
    )
    train_dataset = TensorDataset(X_train_augmented, y_train_augmented)
else:
    train_dataset = TensorDataset(X_train_scaled, y_train)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# MLP

In [9]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
# # Define the MLP model

class MLPClassifier(nn.Module):
    def __init__(self, input_size, hidden_sizes, num_classes, dropout_prob):
        super(MLPClassifier, self).__init__()
        self.layers = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.dropout_prob = dropout_prob
        
        # Input layer
        self.layers.append(nn.Linear(input_size, hidden_sizes[0]))
        self.batch_norms.append(nn.BatchNorm1d(hidden_sizes[0]))

        # Hidden layers
        for i in range(len(hidden_sizes) - 1):
            self.layers.append(nn.Linear(hidden_sizes[i], hidden_sizes[i + 1]))
            self.batch_norms.append(nn.BatchNorm1d(hidden_sizes[i+1]))
        
        # Output layer
        self.layers.append(nn.Linear(hidden_sizes[-1], num_classes))
        
        # Softmax activation for the output layer
        # self.softmax = nn.Softmax(dim=1)
    
    def forward(self, x):
        for i in range(len(self.layers) - 1):
            x = torch.relu(self.batch_norms[i](self.layers[i](x)))
            x = F.dropout(x, p=self.dropout_prob, training=self.training)
        x = self.layers[-1](x)
        # x = self.softmax(x)
        return x

def calculate_confusion_matrix(model, data_loader, num_classes):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X, y in data_loader:
            outputs = model(X)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    
    return confusion_matrix(all_labels, all_preds)

def plot_confusion_matrix(cm, class_names):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()

# Initialize the model
model = MLPClassifier(input_size, hidden_size, num_classes, dropout_rate)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
# Use learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    patience=scheduler_params['patience'],
    factor=scheduler_params['factor'],
    threshold=scheduler_params['threshold'],
    cooldown=scheduler_params['cooldown'],
    min_lr=scheduler_params['min_lr']
)
# Function to predict the class of new data
def predict(model, data):
    model.eval()
    with torch.no_grad():
        output = model(data)
        _, predicted_class = torch.max(output, dim=1)
    return predicted_class

# Function to compute the accuracy
def calculate_accuracy(y_pred, y_true):
    _, predicted = torch.max(y_pred, dim=1)  # Get the index of the max log-probability
    correct = (predicted == y_true).float().sum()
    return correct / y_true.shape[0]

def train_and_evaluate(model, criterion, optimizer, scheduler, train_loader, val_loader, epochs=10, patience=5):
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []

    best_val_loss = float('inf')
    best_model = None
    counter = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_accuracy = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_accuracy += calculate_accuracy(y_pred, y_batch)
        
        train_loss /= len(train_loader)
        train_accuracy /= len(train_loader)
        
        model.eval()
        val_loss = 0
        val_accuracy = 0
        with torch.no_grad():
            for X_val, y_val in val_loader:
                y_val_pred = model(X_val)
                val_loss += criterion(y_val_pred, y_val).item()
                val_accuracy += calculate_accuracy(y_val_pred, y_val)
        
        val_loss /= len(val_loader)
        val_accuracy /= len(val_loader)
        
        if scheduler_params['enabled']:
            scheduler.step(val_loss)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)
        
        print(f'Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}, '
              f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')

        # # Early stopping logic
        # if val_loss < best_val_loss:
        #     best_val_loss = val_loss
        #     best_model = model.state_dict()
        #     counter = 0
        # else:
        #     counter += 1
        #     if counter >= patience:
        #         print(f"Early stopping triggered after {epoch+1} epochs")
        #         model.load_state_dict(best_model)
        #         break
    # cm = calculate_confusion_matrix(model, val_loader, num_classes=num_classes)
    # plot_confusion_matrix(cm, class_names=['Class ' + str(i) for i in range(num_classes)])

    return model, train_losses, val_losses, train_accuracies, val_accuracies

# Use the function
model, train_losses, val_losses, train_accuracies, val_accuracies = train_and_evaluate(
    model, criterion, optimizer, scheduler, train_loader, val_loader, epochs=num_epochs, patience=5
)

# After training, you can also calculate and plot confusion matrix for the test set
# test_cm = calculate_confusion_matrix(model, test_loader, num_classes)
# plot_confusion_matrix(test_cm, class_names=['Class ' + str(i) for i in range(num_classes)])

Epoch 1, Train Loss: 1.2541, Train Accuracy: 0.4674, Validation Loss: 1.0864, Validation Accuracy: 0.5469
Epoch 2, Train Loss: 0.8535, Train Accuracy: 0.7024, Validation Loss: 0.7972, Validation Accuracy: 0.5312
Epoch 3, Train Loss: 0.6858, Train Accuracy: 0.7391, Validation Loss: 0.6630, Validation Accuracy: 0.5781
Epoch 4, Train Loss: 0.5636, Train Accuracy: 0.8084, Validation Loss: 0.5427, Validation Accuracy: 0.8125
Epoch 5, Train Loss: 0.4684, Train Accuracy: 0.8111, Validation Loss: 0.4424, Validation Accuracy: 0.7031
Epoch 6, Train Loss: 0.4765, Train Accuracy: 0.7962, Validation Loss: 0.4169, Validation Accuracy: 0.7656
Epoch 7, Train Loss: 0.4534, Train Accuracy: 0.8071, Validation Loss: 0.3765, Validation Accuracy: 0.6875
Epoch 8, Train Loss: 0.3987, Train Accuracy: 0.8302, Validation Loss: 0.3724, Validation Accuracy: 0.8125
Epoch 9, Train Loss: 0.3886, Train Accuracy: 0.8383, Validation Loss: 0.4583, Validation Accuracy: 0.7969
Epoch 10, Train Loss: 0.3492, Train Accuracy: 